# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will extract, process, and visualize data for clinical and genotype–phenotype correlation analysis.

### Dataset Source
The dataset is defined by a Croissant schema available at:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 clinical dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is an object
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Explore available record sets, fields, and their IDs.

We'll enumerate all record sets and their fields using their `@id`s. This is important for referencing dataset components according to the Croissant schema.

In [ ]:
# Get all record set @ids
record_sets = []
fields_by_recordset = {}
for rs in dataset.record_sets():
    record_sets.append(rs['@id'])
    fields = []
    for field in rs.get('field', []):
        fields.append(field['@id'])
    fields_by_recordset[rs['@id']] = fields

# Display the results
print('Record Sets and Field IDs:')
for rs_id in record_sets:
    print(f"- RecordSet @id: {rs_id}")
    for f_id in fields_by_recordset[rs_id]:
        print(f"    - Field @id: {f_id}")

## 3. Data Extraction
Extract tabular data from each record set into pandas DataFrames.

All entities (record sets, fields) are referenced using their `@id` values from the overview above.

In [ ]:
# Prepare to extract dataframes from each record set
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    # At least some records should exist
    dataframes[rs_id] = pd.DataFrame(records)

# Example: Show columns from the first record set
show_record_set = record_sets[0] if record_sets else None
if show_record_set:
    print(f"Columns for {show_record_set}: {dataframes[show_record_set].columns.tolist()}")
    display(dataframes[show_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Filter records, normalize numeric fields, and group records by attributes. All references are made using `@id` values.

We'll pick a numeric field (such as patient age or hormone level) for demonstration and group by another field (e.g., presence of a clinical feature).

In [ ]:
# Example using the first record set, with known clinical features
rs_id = record_sets[0]
df = dataframes[rs_id]

# Select a numeric field (assume age at diagnosis is present)
# Use @id from overview; here, suppose field @id for age is 'age_at_diagnosis' (replace with actual if different)
numeric_field_id = fields_by_recordset[rs_id][0] # Choose first numeric field for demo
print(f"Using numeric field: {numeric_field_id}")

# Filtering: records with age > 25
threshold = 25
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n{filtered_df.head()}")

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another clinical field
    group_field_id = fields_by_recordset[rs_id][1] if len(fields_by_recordset[rs_id]) > 1 else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found in DataFrame columns.")

## 5. Visualization
Visualize distributions or relationships between numeric clinical features (referenced by their `@id`).

In [ ]:
# Plot histogram of the numeric field for the first record set
if numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=10)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

# Scatter plot: if there are two numeric features
if len(fields_by_recordset[rs_id]) > 2:
    x_field = numeric_field_id
    y_field = fields_by_recordset[rs_id][2]
    if x_field in df.columns and y_field in df.columns:
        plt.figure(figsize=(6,4))
        plt.scatter(df[x_field], df[y_field])
        plt.xlabel(x_field)
        plt.ylabel(y_field)
        plt.title(f'{y_field} vs {x_field}')
        plt.show()

## 6. Conclusion
This notebook demonstrates a stepwise exploration of a FAIR^2 clinical dataset using `mlcroissant`. We loaded record sets, referenced fields by their `@id`, extracted data, filtered and normalized numeric features, and visualized distributions for genotype–phenotype correlation analysis.

**Key Observations:**
- Entity `@id`s make referencing data across steps consistent.
- Data normalization and filtering allow for meaningful comparison within the limited dataset cohort.
- Visualization can highlight clinical heterogeneity in the dataset. 

For further analysis, extend these steps with domain-specific statistical testing or additional FAIR-compliant record sets.